# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuratulainAzhar22/flyrank-ml-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1 — Content Lifecycle: Growing vs Declining

The FlyRank report found that growing pages were younger than declining pages. Growing pages had an average age of about 185 days, while declining pages averaged about 228 days. Their average word counts were almost identical, at about 1.5K words.

**My methodology question:**

How was the growing-versus-declining label defined, and was the label calculated from a time window that was independent of the features used to compare the two groups?

This matters because overlapping the feature window and outcome window could make the observed relationship look stronger than it really is. I would want the label period and feature period to be clearly separated before treating the relationship as predictive evidence.

### Finding 2 — The Freshness Multiplier

The FlyRank report found that pages updated 31–90 days ago had a 5.43:1 growth-to-decline ratio. It also reported a separate comparison for pages older than one year, where recently refreshed pages showed higher observed health and impressions than pages that had not been refreshed as recently.

**My methodology question:**

Does the validation and comparison design support interpreting freshness as a cause of the observed improvement, or could refreshed pages differ from stale pages in other important ways?

This matters because pages selected for refreshing may already have stronger traffic, quality, or strategic value. I would therefore treat the reported relationship as observed evidence unless the comparison design rules out these alternative explanations.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



My Week-5 model was evaluated using its original train/test design. For this audit, I use a client-grouped split so that the same client cannot appear in both training and test data.

This is a more conservative validation design because the model is evaluated on clients that were not used during training.

The purpose is to measure how the model's performance changes under a more realistic validation design, rather than to maximize the metric.

**Week-5 result already measured**

In [5]:


week5_accuracy = 0.989
week5_macro_f1 = 0.9676

print("Week-5 model performance")
print("-------------------------")
print("Accuracy:", week5_accuracy)
print("Macro F1:", week5_macro_f1)

Week-5 model performance
-------------------------
Accuracy: 0.989
Macro F1: 0.9676


**Load a small sample**

In [20]:
import duckdb
import pandas as pd
from google.colab import userdata

# Get your Hugging Face read token from Colab Secrets
HF_TOKEN = userdata.get("My_Read_Token")

if not HF_TOKEN:
    raise ValueError(
        "My_Read_Token was not found. "
        "Add your Hugging Face token to Colab Secrets with the name My_Read_Token."
    )

con = duckdb.connect()

# Give DuckDB permission to access Hugging Face
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

sample = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('{march_path}')
    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions > 0
        AND gsc_avg_position IS NOT NULL
    USING SAMPLE reservoir(500000 ROWS)
    REPEATABLE (42)
""").df()

print("Rows:", len(sample))
print("Clients:", sample["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 179598
Clients: 44


**Create the same baseline-style target**

In [21]:
# Calculate CTR
sample["ctr"] = (
    sample["gsc_clicks"] / sample["gsc_impressions"]
)

# Position buckets
sample["position_bucket"] = pd.cut(
    sample["gsc_avg_position"],
    bins=[-float("inf"), 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

# Position-level CTR benchmark
benchmarks = (
    sample.groupby("position_bucket", observed=True)
    .apply(
        lambda x: x["gsc_clicks"].sum() /
                  x["gsc_impressions"].sum()
    )
    .rename("benchmark_ctr")
)

sample = sample.join(
    benchmarks,
    on="position_bucket"
)

sample["ctr_gap"] = (
    sample["benchmark_ctr"] - sample["ctr"]
)

# Baseline score
sample["score"] = 0

sample.loc[
    (sample["gsc_impressions"] >= 100) &
    (sample["ctr"] < sample["benchmark_ctr"]),
    "score"
] = 2

sample.loc[
    (sample["gsc_impressions"] < 100) &
    (sample["ctr"] < sample["benchmark_ctr"]),
    "score"
] = 1

print(sample["score"].value_counts().sort_index())

score
0     19142
1    138189
2     22267
Name: count, dtype: int64


/tmp/ipykernel_664/1070709087.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


**Client-grouped split**

In [22]:
from sklearn.model_selection import GroupShuffleSplit

# Features used by the simple model
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

X = sample[feature_cols].copy()
y = sample["score"].copy()
groups = sample["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 168431
Test rows: 11167
Training clients: 35
Test clients: 9
Client overlap: 0


**Train the desision tree**

In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

grouped_pred = dt_model.predict(X_test)

grouped_accuracy = accuracy_score(
    y_test,
    grouped_pred
)

grouped_macro_f1 = f1_score(
    y_test,
    grouped_pred,
    average="macro"
)

print("Honest client-grouped performance")
print("----------------------------------")
print("Accuracy:", round(grouped_accuracy, 4))
print("Macro F1:", round(grouped_macro_f1, 4))

Honest client-grouped performance
----------------------------------
Accuracy: 0.9997
Macro F1: 0.9965


**Before vs After**

In [24]:
comparison = pd.DataFrame({
    "Evaluation": [
        "Week-5 original split",
        "Week-6 client-grouped split"
    ],
    "Accuracy": [
        week5_accuracy,
        grouped_accuracy
    ],
    "Macro F1": [
        week5_macro_f1,
        grouped_macro_f1
    ]
})

comparison

,Evaluation,Accuracy,Macro F1
0,Week-5 original split,0.989000,0.967600
1,Week-6 client-grouped split,0.999731,0.996452


### Before vs after interpretation

The Week-5 model achieved a measured Accuracy of 0.989 and Macro F1 of 0.9676 under the original evaluation.

The Week-6 client-grouped evaluation provides a more conservative test because no client appears in both training and test sets.

The difference between the two evaluations is treated as an observed validation effect. I do not interpret the grouped result as proof of real-world performance because the evaluation still uses a limited sample and the target is derived from the baseline rule.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I audited the features used by the Week-5-style model for identifiers, target-derived variables, and variables that would not be available independently of the target.

Client and content identifiers are used for grouping and splitting only. They are not model features.

I also checked whether the target is mathematically derived from the model inputs. This is important because a model can achieve a high score simply by learning the rule that generated the target rather than learning a genuinely independent future outcome.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features actually used by the model

print("Model features:")
for col in feature_cols:
    print("-", col)

# Identifier leakage check
identifier_cols = [
    "client_hash_id",
    "content_hash_id"
]

identifier_leakage = [
    col for col in identifier_cols
    if col in feature_cols
]

print("\nIdentifier columns used as features:")
print(identifier_leakage)

Model features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position

Identifier columns used as features:
[]


In [26]:
target_derived_features = [
    "score",
    "reason_code",
    "action",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

found_leakage = [
    col for col in target_derived_features
    if col in feature_cols
]

print("Target-derived / forbidden features found:")
print(found_leakage)

Target-derived / forbidden features found:
[]


### Leakage finding

The audit found no identifier or explicit target column inside the feature list.

However, there is an important methodological limitation: the `score` target used in this reconstruction is generated from the same current-period search signals used as model inputs. Therefore, the Decision Tree is learning to reproduce a deterministic baseline rule rather than independently predicting a future outcome.

The high measured model performance should therefore not be interpreted as evidence that the model predicts future content decline or improvement. It mainly shows that the model can reproduce the existing rule.

**Failure examples**

I inspected cases where the Decision Tree prediction did not match the baseline-derived target. These examples are useful because they show where the model does not perfectly reproduce the rule.

The errors should not be interpreted as proof that the underlying content is good or bad. They only show disagreement between the model prediction and the rule-derived target.

In [19]:
errors = sample.iloc[test_idx].copy()

errors["predicted_score"] = grouped_pred
errors["actual_score"] = y_test.values

errors = errors[
    errors["predicted_score"] != errors["actual_score"]
]

print("Number of errors:", len(errors))

errors[
    [
        "client_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "actual_score",
        "predicted_score"
    ]
].head(10)

Number of errors: 76


,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,actual_score,predicted_score
25036,client_1a730cb2640a1abf,259,3,0.521236,0,2
25037,client_1a730cb2640a1abf,258,3,2.941860,0,2
25042,client_1a730cb2640a1abf,306,2,1.068627,0,2
25298,client_1a730cb2640a1abf,252,3,1.662698,0,2
25407,client_1a730cb2640a1abf,265,5,2.290566,0,2
25418,client_1a730cb2640a1abf,253,5,1.426877,0,2
25427,client_1a730cb2640a1abf,257,4,1.887160,0,2
25428,client_1a730cb2640a1abf,293,5,1.883959,0,2
25481,client_1a730cb2640a1abf,253,7,2.972332,0,2
25597,client_0fa64a184f18a4a0,1097,3,3.369189,2,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Decision Tree successfully predicts which pages should be prioritized for review.

### Rewritten claim

The Decision Tree showed high measured agreement with the baseline-derived target on the evaluated data. However, because the target was generated from the same current-period search signals used as model inputs, this result mainly demonstrates that the model can reproduce the existing prioritization rule. It should therefore be treated as directional decision-support rather than evidence of independent future-performance prediction.

The client-grouped validation provides a more conservative evaluation of generalization across clients, but further work with an independently defined future outcome would be needed before making a stronger predictive claim.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.